# Hawkes Process (Self-Exciting) Generator

Generate self-exciting point process time series using the Hawkes process model.
In a Hawkes process, past events increase the probability of future events, creating clustering behavior.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

from synforecast.generators import HawkesProcessGenerator

## 1. Event Counts (Default Output)

Generate event count time series with a baseline intensity and self-excitation.

In [ ]:
params = {
    "min_length": 200,
    "max_length": 200,
    "freq": "h",
    "baseline_intensity": 1.0,
    "excitation_amplitude": 0.5,
    "decay_rate": 2.0,
    "output_type": "counts",
    "seed": 42,
}

generator = HawkesProcessGenerator(engine="polars", **params)
df = generator.generate(n_series=3)

print(f"Generated {df['unique_id'].n_unique()} time series")
print(f"Total observations: {len(df)}")
print(f"Total events: {df['y'].sum()}")

stats = df.group_by("unique_id").agg(
    [
        pl.col("y").sum().alias("total_events"),
        pl.col("y").mean().alias("mean_per_hour"),
        pl.col("y").max().alias("max_in_hour"),
    ]
)
stats

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df["unique_id"].unique().to_list():
    series = df.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Event Count")
ax.set_title("Hawkes Process - Event Counts")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Intensity Process

Output the underlying intensity function instead of event counts.

In [ ]:
intensity_gen = HawkesProcessGenerator(engine="polars", 
    **{
        "min_length": 100,
        "max_length": 100,
        "freq": "h",
        "baseline_intensity": 0.5,
        "excitation_amplitude": 0.3,
        "decay_rate": 1.0,
        "output_type": "intensity",
        "seed": 42,
    }
)
intensity_df = intensity_gen.generate(n_series=1)

print(f"Intensity range: [{intensity_df['y'].min():.3f}, {intensity_df['y'].max():.3f}]")
print(f"Mean intensity: {intensity_df['y'].mean():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in intensity_df["unique_id"].unique().to_list():
    series = intensity_df.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Intensity")
ax.set_title("Hawkes Process - Intensity Function")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Model Information and Stability

Inspect model parameters including the branching ratio which determines stability.

In [ ]:
info = generator.get_model_info()
print(f"Baseline intensity (mu): {info['baseline_intensity']}")
print(f"Excitation amplitude (alpha): {info['excitation_amplitude']}")
print(f"Decay rate (beta): {info['decay_rate']}")
print(f"Branching ratio (alpha/beta): {info['branching_ratio']:.3f}")
print(f"Expected cluster size: {info['expected_cluster_size']:.3f}")
print(f"Process is stable: {info['is_stable']}")

## 4. Raw Event Simulation

Directly simulate event arrival times and their associated intensities.

In [ ]:
event_times, intensities = generator.simulate_with_events(time_horizon=50.0)
print(f"Simulated {len(event_times)} events over 50 time units")
print(f"Event rate: {len(event_times) / 50:.2f} events/unit time")
print(f"First 10 event times: {event_times[:10].round(3)}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(event_times, intensities, alpha=0.8)
axes[0].set_ylabel("Intensity")
axes[0].set_title("Hawkes Process - Event Arrivals and Intensity")
axes[1].eventplot([event_times], lineoffsets=0.5, linelengths=0.8, colors="tab:red")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Events")
axes[1].set_yticks([])
plt.tight_layout()
plt.show()

## 5. Power-Law Kernel

Use a power-law decay kernel instead of the default exponential kernel for longer memory effects.

In [ ]:
power_law_gen = HawkesProcessGenerator(engine="polars", 
    **{
        "min_length": 100,
        "max_length": 100,
        "freq": "h",
        "kernel": "power_law",
        "power_law_exponent": 1.5,
        "baseline_intensity": 0.5,
        "excitation_amplitude": 0.2,
        "seed": 42,
    }
)
power_law_df = power_law_gen.generate(n_series=1)
print(f"Power-law kernel total events: {power_law_df['y'].sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in power_law_df["unique_id"].unique().to_list():
    series = power_law_df.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Event Count")
ax.set_title("Hawkes Process - Power-Law Kernel")
ax.legend()
plt.tight_layout()
plt.show()